# LLM4Decompile お試し — やまぶきR (yabu_r64.exe)

GitHub Issue #14 / AutoFor/cornix-oyayubi のお試し実験。
**目的**: `LLM4Binary/llm4decompile-6.7b-v1.5` で、逆アセンブルした関数を擬似Cに復元し、
ツールが動くか・可読なCが出るか・同時打鍵の判定ロジックの手掛かりが出るかを確認する。

## 使い方
1. メニュー **ランタイム → ランタイムのタイプを変更 → T4 GPU** を選ぶ。
2. 上から順にセルを実行する（モデルDLに数分）。
3. 出力Cを `docs/reference/yamabuki-r/llm4decompile-trial.md` の該当欄に貼る。

## 対象関数（yabu_r64.exe から抽出済み・下に直書き）
| キー | 関数 | 位置づけ |
|---|---|---|
| `fcn_140001b30.asm` | 小リーフ関数(103B) | **感触確認** — まず動くか |
| `fcn_140004fc0_security_cookie.asm` | `__security_init_cookie`(179B) | **精度検証** — 正解ソースが公知（MSVC CRT）。復元Cの正しさを厳密に採点できる |
| `fcn_140001000_msgloop.asm` | GetMessageW メッセージループ(477B) | **本命** — フックからPostMessageWされたキーイベントを処理する側。判定ロジックの手掛かりを探す |

> 注: LLM4Decompile は gcc/Linux ELF 中心の学習。対象は MSVC/Windows PE なので**分布外**であり、
> 出力は粗くなりうる。お試しなのでそれ込みで感触を見る。

## 1. GPU 確認（T4 が出ればOK）

In [ ]:
!nvidia-smi

## 2. 依存導入

In [ ]:
!pip -q install 'transformers>=4.41' accelerate bitsandbytes
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

## 3. 入力ASM（リポジトリから抽出済みを直書き）
GitHub から取り直したい場合は下の `USE_GITHUB=True` にする。

In [ ]:
ASM = {
    "fcn_140001000_msgloop.asm": "<msgloop_judge>:\nmov %rbx,0x8(%rsp)\nmov %rbp,0x10(%rsp)\nmov %rsi,0x20(%rsp)\npush %rdi\nsub $0x80,%rsp\nmov 0xd0a2(%rip),%rax\nxor %rsp,%rax\nmov %rax,0x78(%rsp)\nmov %rcx,%rsi\nmov %r8,%rdi\nxor %eax,%eax\nmov %r8,%rdx\nxor %ebx,%ebx\nor $0xffffffffffffffff,%rbp\nmov %rbp,%rcx\nmovq $0x7,0x70(%rsp)\nmov %rbx,0x68(%rsp)\nrepnz scas %es:(%rdi),%ax\nmov %bx,0x58(%rsp)\nnot %rcx\nlea -0x1(%rcx),%r8\nlea 0x50(%rsp),%rcx\ncall 0x1400013a0\nmov 0x68(%rsp),%r8\nmov 0x70(%rsp),%r9\nmov 0x58(%rsp),%r10\ncmp $0x8,%r9\nmov $0x12,%edi\nlea 0x58(%rsp),%rdx\nlea 0xafc3(%rip),%rcx\ncmovae %r10,%rdx\ncmp %rdi,%r8\ncmovb %r8,%rdi\ntest %rdi,%rdi\nje 0x1400010ab\nmovzwl (%rcx),%eax\ncmp %ax,(%rdx)\njne 0x1400010f7\nadd $0x2,%rdx\nadd $0x2,%rcx\nsub $0x1,%rdi\njne 0x140001095\nmov %ebx,%edi\nmovslq %edi,%rax\ntest %edi,%edi\njne 0x1400010c3\ncmp $0x12,%r8\njb 0x1400011a7\nsetne %bl\nmov %ebx,%eax\ntest %eax,%eax\njne 0x1400011a7\nmov %rsi,0xf07e(%rip)\ncall 0x1400011e0\ntest %al,%al\njne 0x140001104\ncmpq $0x8,0x70(%rsp)\njb 0x1400010ed\nmov 0x58(%rsp),%rcx\ncall 0x140002148\nmov $0x1,%eax\njmp 0x1400011b7\ncmp %ax,(%rdx)\nmov $0x1,%edi\ncmovb %ebp,%edi\njmp 0x1400010ad\nlea 0xafbd(%rip),%rcx\ncall *0x8f1f(%rip)\nmov %rax,%rdi\ntest %rax,%rax\nje 0x140001143\nlea 0xafc0(%rip),%rdx\nmov %rax,%rcx\ncall *0x8f0f(%rip)\ntest %rax,%rax\nje 0x14000113a\nmov $0x1,%edx\nmov $0x8005,%ecx\ncall *%rax\nmov %rdi,%rcx\ncall *0x8efd(%rip)\nlea 0x20(%rsp),%rcx\nxor %r9d,%r9d\nxor %r8d,%r8d\nxor %edx,%edx\ncall *0x90da(%rip)\ntest %eax,%eax\nje 0x14000118d\nnopw 0x0(%rax,%rax,1)\nlea 0x20(%rsp),%rcx\ncall *0x909d(%rip)\nlea 0x20(%rsp),%rcx\ncall *0x90da(%rip)\nlea 0x20(%rsp),%rcx\nxor %r9d,%r9d\nxor %r8d,%r8d\nxor %edx,%edx\ncall *0x90a7(%rip)\ntest %eax,%eax\njne 0x140001160\ncmpq $0x8,0x70(%rsp)\nmov 0x30(%rsp),%edi\njb 0x1400011a3\nmov 0x58(%rsp),%rcx\ncall 0x140002148\nmov %edi,%eax\njmp 0x1400011b7\ncmp $0x8,%r9\njb 0x1400011b5\nmov %r10,%rcx\ncall 0x140002148\nxor %eax,%eax\nmov 0x78(%rsp),%rcx\nxor %rsp,%rcx\ncall 0x140002090\nlea 0x80(%rsp),%r11\nmov 0x10(%r11),%rbx\nmov 0x18(%r11),%rbp\nmov 0x28(%r11),%rsi\nmov %r11,%rsp\npop %rdi\nret\n",
    "fcn_140001b30.asm": "<sanity_leaf>:\nsub $0x98,%rsp\nmovq $0xfffffffffffffffe,0x20(%rsp)\nlea 0x8859(%rip),%rdx\nlea 0x28(%rsp),%rcx\ncall 0x140001c9c\nnop\nlea 0x50(%rsp),%rcx\ncall 0x140002150\nnop\nlea 0x87d4(%rip),%rax\nmov %rax,0x50(%rsp)\nlea 0x28(%rsp),%rdx\nlea 0x68(%rsp),%rcx\ncall 0x140001f08\nnop\nlea 0x87d0(%rip),%rax\nmov %rax,0x50(%rsp)\nlea 0xb4d4(%rip),%rdx\nlea 0x50(%rsp),%rcx\ncall 0x1400026e8\nint3\n",
    "fcn_140004fc0_security_cookie.asm": "<security_init_cookie>:\nmov %rbx,0x18(%rsp)\npush %rdi\nsub $0x20,%rsp\nmov 0x90ef(%rip),%rax\nandq $0x0,0x30(%rsp)\nmovabs $0x2b992ddfa232,%rdi\ncmp %rdi,%rax\nje 0x140004ff2\nnot %rax\nmov %rax,0x90d8(%rip)\njmp 0x140005068\nlea 0x30(%rsp),%rcx\ncall *0x5193(%rip)\nmov 0x30(%rsp),%rbx\ncall *0x5180(%rip)\nmov %eax,%r11d\nxor %r11,%rbx\ncall *0x5144(%rip)\nmov %eax,%r11d\nxor %r11,%rbx\ncall *0x5160(%rip)\nlea 0x38(%rsp),%rcx\nmov %eax,%r11d\nxor %r11,%rbx\ncall *0x5147(%rip)\nmov 0x38(%rsp),%r11\nxor %rbx,%r11\nmovabs $0xffffffffffff,%rax\nand %rax,%r11\nmovabs $0x2b992ddfa233,%rax\ncmp %rdi,%r11\ncmove %rax,%r11\nmov %r11,0x9062(%rip)\nnot %r11\nmov %r11,0x9060(%rip)\nmov 0x40(%rsp),%rbx\nadd $0x20,%rsp\npop %rdi\nret\n"
}

USE_GITHUB = False  # True にすると raw.githubusercontent.com から取得
if USE_GITHUB:
    import urllib.request
    base = 'https://raw.githubusercontent.com/AutoFor/cornix-oyayubi/main/tools/decompile/inputs/'
    for name in list(ASM):
        ASM[name] = urllib.request.urlopen(base + name).read().decode()

for name, body in ASM.items():
    print(f'--- {name}  ({body.count(chr(10))} lines) ---')
    print(body[:200])

## 4. モデルロード
T4(Turing) は bf16 非対応なので **fp16**。16GBに載らずOOMになったら、
次セルの `LOAD_IN_4BIT = True` にして再実行する。

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL = 'LLM4Binary/llm4decompile-6.7b-v1.5'
LOAD_IN_4BIT = False  # OOM時は True

tok = AutoTokenizer.from_pretrained(MODEL)
if LOAD_IN_4BIT:
    from transformers import BitsAndBytesConfig
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map='auto')
else:
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16).cuda()
model.eval()
print('loaded:', MODEL, '4bit' if LOAD_IN_4BIT else 'fp16')

## 5. デコンパイル実行
LLM4Decompile v1.5 のプロンプト形式に合わせて ASM を渡し、擬似Cを生成する。

In [ ]:
def decompile(asm_text: str, max_new_tokens: int = 2048) -> str:
    prompt = f'# This is the assembly code:\n{asm_text}\n# What is the source code?\n'
    inputs = tok(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id)
    gen = out[0][inputs['input_ids'].shape[1]:]
    return tok.decode(gen, skip_special_tokens=True)

results = {}
for name, body in ASM.items():
    print('=' * 70)
    print('#', name)
    print('=' * 70)
    c = decompile(body)
    results[name] = c
    print(c)
    print()

## 6.（任意）Opus 4.8 と比較
同じ ASM を Anthropic API に渡し、可読性を見比べる。
Colab の秘密（🔑アイコン）に `ANTHROPIC_API_KEY` を登録してから実行する。無ければスキップでOK。

In [ ]:
SKIP_OPUS = True  # 比較する場合は False
if not SKIP_OPUS:
    !pip -q install anthropic
    import os, anthropic
    try:
        from google.colab import userdata
        os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    except Exception:
        pass
    client = anthropic.Anthropic()
    for name, body in ASM.items():
        msg = client.messages.create(
            model='claude-opus-4-8',
            max_tokens=2048,
            messages=[{'role': 'user', 'content':
                'これはWindows x86-64のディスアセンブルです。読みやすいC疑似コードに復元し、'
                '関数の役割を1行で述べてください。\n\n' + body}],
        )
        print('=' * 70); print('#', name, '(Opus 4.8)'); print('=' * 70)
        print(msg.content[0].text); print()

## 7. 結果の記録
出力を `docs/reference/yamabuki-r/llm4decompile-trial.md` の各欄に貼る。観点:
- 構文的に妥当なCか / 変数名・制御構造は追えるか
- `__security_init_cookie` は**正解が公知**（QueryPerformanceCounter でカナリア種を作る）。
  復元Cがそれに近いほどデコンパイラの精度が高い。
- メッセージループで、キーイベントのタイムスタンプ比較・range%判定の痕跡が見えるか。